## 1. Setup & Imports

In [2]:
import pandas as pd
import os 
import re

## 2. Load Data

In [3]:
parquet_path = 'data/finaccess_2024.parquet'

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print("Loaded from Parquet (fast path). Shape:", df.shape)
else:
    excel_path = '../data/2024_Finaccess_Publicdata.xlsx'
    df = pd.read_excel(excel_path, sheet_name='2024_Finaccess_Publicdata')

    # Parquet requires consistent column types, so convert mixed-type columns to string
    for col in df.select_dtypes(include=['Object']).columns:
        df[col] = df[col].astype(str)

    os.makedirs('data', exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    print("Loaded from Excel and saved to Parquet for next time. Shape:", df.shape)

Loaded from Parquet (fast path). Shape: (20871, 3816)


## 3. Load Codebook

In [4]:
vars_df = pd.read_excel('../data/2024_Finaccess_Publicdata.xlsx', sheet_name='Variables')
values_df = pd.read_excel('../data/2024_Finaccess_Publicdata.xlsx', sheet_name='Values')

print("Variables sheet shape:", vars_df.shape)
print("Values sheet shape:", values_df.shape)




Variables sheet shape: (3816, 7)
Values sheet shape: (215327, 1)


### 3.1 Search for Relevant Variables

In [5]:
keywords = ['loan', 'credit', 'borrow', 'debt', 'repay', 'default',
            'arrears', 'overdue', 'behind', 'missed', 'struggl', 'difficult',
            'unable to pay', 'asset', 'machinery', 'equipment', 'solar',
            'mobile money', 'lend', 'installment']

mask = (vars_df['varlab'].str.contains('|'.join(keywords), case=False, na=False) |
        vars_df['name'].str.contains('|'.join(keywords), case=False, na=False))

relevant_vars = vars_df[mask][['name', 'varlab']]

os.makedirs('data', exist_ok=True)
relevant_vars.to_csv('data/relevant_vars.csv', index=False)
print(f"Found {len(relevant_vars)} relevant variables — saved to CSV")

Found 718 relevant variables — saved to CSV


## 4. Identify Target Variable (E2E)

In [6]:
target_col = 'E2E'

print("Class balance for E2E (repayment status):")
print(df[target_col].value_counts(dropna=False))

Class balance for E2E (repayment status):
E2E
nan                                     7720
I did not default/always pay on time    6160
Paid late/Missed a payment/Paid less    5015
Didn't pay at all                       1936
Donâ€™t know                              37
Refused to Answer                          3
Name: count, dtype: int64


## 5. Filter to Active Credit Users

In [7]:
valid_responses = [
    'I did not default/always pay on time',
    'Paid late/Missed a payment/Paid less',
    "Didn't pay at all"
] 

modeling_df = df[df[target_col].isin(valid_responses)].copy()

print(f"Active credit users (valid E2E response): {modeling_df.shape[0]}")
print(modeling_df[target_col].value_counts())

Active credit users (valid E2E response): 13111
E2E
I did not default/always pay on time    6160
Paid late/Missed a payment/Paid less    5015
Didn't pay at all                       1936
Name: count, dtype: int64


## 6. Feature Selection

In [8]:
# Binary target: 0 = always paid on time, 1 = any repayment difficulty
modeling_df['defaulted'] = modeling_df[target_col].apply(
    lambda x: 0 if x == 'I did not default/always pay on time' else 1
)

# Loan-type usage columns (confirmed main C1 loan-product columns, not sub-columns)
loan_cols = ['C1_14', 'C1_15', 'C1_16', 'C1_17', 'C1_18', 'C1_19',
             'C1_20', 'C1_21', 'C1_22', 'C1_23']

# Asset ownership (purchased machinery/equipment for business or farming)
asset_cols = ['B1L__4', 'B1L__5']

# Demographic / geographic context
demo_cols = ['county', 'A07', 'A08']  # A07/A08 confirmed earlier as sex/age-type fields

feature_cols = loan_cols + asset_cols + demo_cols
print(f"Total candidate features selected: {len(feature_cols)}")
print(modeling_df[feature_cols].isnull().sum())

Total candidate features selected: 15
C1_14     0
C1_15     0
C1_16     0
C1_17     0
C1_18     0
C1_19     0
C1_20     0
C1_21     0
C1_22     0
C1_23     0
B1L__4    0
B1L__5    0
county    0
A07       0
A08       0
dtype: int64


## 7. Exploratory Analysis

In [14]:
# Overall class balance
print("--- Overall class balance ---")
print(modeling_df['defaulted'].value_counts(normalize=True).round(3))
print()

pd.set_option('display.max_rows', None)
# Default rate by loan type, with sample sizes (not just percentages)
results = []
for col in loan_cols:
    counts = modeling_df.groupby(col)['defaulted'].agg(['mean', 'count'])
    counts.columns = ['default_rate', 'n']
    counts.index.name = 'usage_status'   # <-- new line: rename BEFORE reset_index
    counts['loan_type'] = col
    results.append(counts.reset_index())

default_by_loan = pd.concat(results, ignore_index=True)
default_by_loan = default_by_loan.rename(columns={default_by_loan.columns[0]: 'usage_status'})
default_by_loan

--- Overall class balance ---
defaulted
1    0.53
0    0.47
Name: proportion, dtype: float64



,usage_status,default_rate,n,loan_type
0,Currently use,0.227273,22,C1_14
1,Never used,0.531231,13032,C1_14
2,Used to use,0.403509,57,C1_14
3,Currently use,0.675642,1597,C1_15
4,Never used,0.505644,9922,C1_15
5,Not stated,0.000000,1,C1_15
6,Used to use,0.537398,1591,C1_15
7,Currently use,0.606149,3415,C1_16
8,Never used,0.499381,7273,C1_16
9,Used to use,0.515477,2423,C1_16
